# Ejercicio 1 — Detección de curvatura: $3^2$ vs. $2^2$ (R)

**Objetivo.** Comparar la capacidad de un diseño $2^2$ y un $3^2$ para detectar curvatura.
Ajustar modelos lineal y cuadrático y diagnosticar falta de ajuste.

**Factores:** Temperatura ($A$: 60/75/90 °C) y Concentración ($B$: 10/15/20 g/L)
**Respuesta:** Rendimiento de reacción (%)

In [ ]:
library(dplyr)
library(ggplot2)
library(rsm)

df <- read.csv('../../datos/rendimiento-reaccion-3k.csv')
print(df)

## 1. Subconjunto $2^2$: solo los 4 vértices

In [ ]:
df22 <- df %>% filter(x1 %in% c(-1, 1), x2 %in% c(-1, 1))
cat('Corridas del subconjunto 2^2:\n')
print(df22[, c('x1', 'x2', 'rendimiento')])

modelo_lineal <- lm(rendimiento ~ x1 + x2 + x1:x2, data = df22)
summary(modelo_lineal)

## 2. Modelo cuadrático — $3^2$ completo

In [ ]:
modelo_cuad <- lm(rendimiento ~ x1 + x2 + I(x1^2) + I(x2^2) + x1:x2, data = df)
summary(modelo_cuad)
anova(modelo_cuad)

## 3. Diagnóstico de curvatura

In [ ]:
pred_centro <- predict(modelo_lineal, newdata = data.frame(x1 = 0, x2 = 0))
y_obs_centro <- df %>% filter(x1 == 0, x2 == 0) %>% pull(rendimiento)

cat(sprintf('Predicción lineal en (0,0): %.2f\n', pred_centro))
cat(sprintf('Observación real en (0,0):  %.2f\n', y_obs_centro))
cat(sprintf('Diferencia (curvatura):     %.2f\n', y_obs_centro - pred_centro))

## 4. Superficie de respuesta con `rsm`

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)
modelo_rsm <- rsm(rendimiento ~ SO(x1, x2), data = df)
summary(modelo_rsm)

par(mfrow = c(1, 2))
contour(modelo_rsm, ~ x1 + x2, image = TRUE,
        col.image = terrain.colors(20),
        xlab = 'x1 (Temperatura)', ylab = 'x2 (Concentración)',
        main = 'Curvas de nivel')
persp(modelo_rsm, ~ x1 + x2,
      col = terrain.colors(20), contour = 'colors',
      theta = -30, phi = 25, main = 'Superficie 3D')

## 5. Conclusión

- El **$2^2$** estima solo efectos lineales; la curvatura queda completamente oculta.
- El **$3^2$** detecta curvatura significativa en **ambos** factores:
  $\hat\beta_{11} < 0$ (temperatura, $p \approx 0.02$) y $\hat\beta_{22} < 0$
  (concentración, $p \approx 0.001$). La concentración muestra la curvatura más pronunciada,
  pero la temperatura también contribuye de forma estadísticamente significativa.
- La diferencia entre la predicción lineal y el valor observado en el centro cuantifica
  la magnitud total de la curvatura.
- Próximo paso: **CCD** o **Box-Behnken** para localizar el máximo con precisión.